# Train DistilBERT 3-Class Issue Classifier (Maintainer's Copilot)

**End-to-end DL workflow — fetch, label-merge, split, train, model card, verify.**

Runs on Colab (T4 GPU, free tier). Produces `artifacts/classifier/best/` containing weights + tokenizer + `model_card.json`, ready to be loaded by `app.ml.classifier.load_classifier()` in the backend.

## Why this is a notebook, not Python in the repo
Training is offline, runs once on a GPU. Inference is online, runs in production. The contract between the two is `model_card.json` — the only file that crosses the boundary. The backend has zero training-side imports.

## Required contract with backend (`app/ml/classifier.py`)
The model card MUST contain:
* `classes`: `["bug", "feature", "support"]` (exact order — drift = refuse to boot)
* `model_sha256`: `sha256:<hex>` of all weight files sorted by name
* `version`: semver string
* `hyperparameters.max_length`: int (used by tokenizer at inference)

## Dataset rationale (also in `docs/DECISIONS.md`)
MONAI closed-issue label counts: `bug=337`, `feature_request=535`, `documentation=28`, `questions=250`.
A 28-example `documentation` class yields a ~5-example test slice — F1 on 5 examples is noise (1 error ≈ 20-pt swing).
Merge `documentation + questions → support` (maintainer's routing is identical for both). Final balanced 3-class problem: `bug=337`, `feature=535`, `support=278`.

## 0. Setup — install deps, mount Drive

In [ ]:
# Pin to versions used in backend/pyproject.toml for inference parity.
!pip install -q \
    "torch>=2.12.0" \
    "transformers>=5.8.1" \
    "datasets>=4.8.5" \
    "evaluate>=0.4.6" \
    "accelerate>=1.13.0" \
    "scikit-learn>=1.8.0" \
    "pydantic>=2.13.4" \
    "httpx>=0.28.1"

import torch
print(f"torch={torch.__version__}  cuda={torch.cuda.is_available()}  device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# Mount Google Drive — data in, artifacts out.
# If you'd rather work fully ephemerally, comment this out and use /content paths.
from google.colab import drive  # type: ignore[import-not-found]
drive.mount('/content/drive')

from pathlib import Path
WORK_DIR = Path('/content/drive/MyDrive/maintainers-copilot')
DATA_DIR = WORK_DIR / 'data'
ARTIFACTS_DIR = WORK_DIR / 'artifacts' / 'classifier'
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print('WORK_DIR =', WORK_DIR)

In [ ]:
# Optional: GitHub PAT raises rate limit from 60 → 5000 req/hour.
# Stored as a Colab secret. Skip if you can complete the fetch under 60 calls (paginated).
import os
try:
    from google.colab import userdata  # type: ignore[import-not-found]
    os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN')
    print('GITHUB_TOKEN loaded from Colab secrets')
except Exception:
    print('No GITHUB_TOKEN — using anonymous rate limit (60/hr)')

## 1. Fetch MONAI closed issues from GitHub

In [ ]:
import json
import time
from datetime import datetime
from typing import Any

import httpx

REPO = 'Project-MONAI/MONAI'
TARGET_LABELS = ('bug', 'feature request', 'documentation', 'questions')
RAW_PATH = DATA_DIR / 'raw_issues.jsonl'


def fetch_closed_issues(repo: str, force: bool = False) -> list[dict[str, Any]]:
    """Page through GitHub API; cache to RAW_PATH so we don't refetch on rerun."""
    if RAW_PATH.exists() and not force:
        with RAW_PATH.open() as fh:
            rows = [json.loads(line) for line in fh]
        print(f'cache hit: {len(rows)} issues from {RAW_PATH}')
        return rows

    headers = {'Accept': 'application/vnd.github+json'}
    if token := os.environ.get('GITHUB_TOKEN'):
        headers['Authorization'] = f'Bearer {token}'

    issues: list[dict[str, Any]] = []
    page = 1
    with httpx.Client(timeout=30.0, headers=headers) as client:
        while True:
            r = client.get(
                f'https://api.github.com/repos/{repo}/issues',
                params={'state': 'closed', 'per_page': 100, 'page': page},
            )
            r.raise_for_status()
            batch = r.json()
            if not batch:
                break
            # Skip PRs (GitHub API returns them in /issues with a pull_request field).
            issues.extend(i for i in batch if 'pull_request' not in i)
            print(f'page {page}: +{len(batch)}  total={len(issues)}')
            page += 1
            time.sleep(0.2)  # be polite

    # Slim & save.
    slim = [
        {
            'id': i['id'],
            'number': i['number'],
            'title': i['title'],
            'body': i.get('body'),
            'labels': [l['name'] for l in i.get('labels', [])],
            'created_at': i['created_at'],
            'closed_at': i['closed_at'],
        }
        for i in issues
    ]
    with RAW_PATH.open('w') as fh:
        for row in slim:
            fh.write(json.dumps(row) + '\n')
    print(f'saved {len(slim)} issues → {RAW_PATH}')
    return slim


raw_issues = fetch_closed_issues(REPO)

## 2. Label mapping — 4 GitHub labels → 3 canonical classes

**Merge rationale (also in `docs/DECISIONS.md`):** `documentation` has only 28 closed issues → ~5 test examples after a 70/15/15 split. Per-class F1 on 5 examples is statistical noise (1 misclassification ≈ 20-pt swing). For a maintainer, the routing decision for `documentation` and `questions` is identical ("point user to the right doc/FAQ"), so merging them into `support` is semantically valid and gives a balanced, defensible 3-class problem.

In [ ]:
from collections import Counter

# Must match backend/app/domain/issue.py — drift is checked by load_classifier().
LABEL_MAP: dict[str, str] = {
    'bug': 'bug',
    'feature request': 'feature',
    'documentation': 'support',
    'questions': 'support',
}
CLASS_NAMES: tuple[str, ...] = ('bug', 'feature', 'support')
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
_TARGET_LABELS = frozenset(LABEL_MAP.keys())


def resolve_label(labels: list[str]) -> str | None:
    """Return canonical class or None if 0 / >1 target labels present."""
    mapped = {LABEL_MAP[lbl] for lbl in labels if lbl in _TARGET_LABELS}
    return mapped.pop() if len(mapped) == 1 else None


def build_row(raw: dict[str, Any]) -> dict[str, Any] | None:
    label = resolve_label(raw['labels'])
    if label is None or raw.get('closed_at') is None:
        return None
    text = f"{raw['title']}\n\n{raw.get('body') or ''}".strip()
    return {
        'id': raw['id'],
        'number': raw['number'],
        'text': text,
        'label': label,
        'label_idx': CLASS_TO_IDX[label],
        'closed_at': raw['closed_at'],
    }


# Show raw label distribution before mapping.
raw_label_counts = Counter(lbl for i in raw_issues for lbl in i['labels'] if lbl in _TARGET_LABELS)
print('Raw target-label counts (issues may have multiple labels):')
for lbl, n in raw_label_counts.most_common():
    print(f'  {lbl:<20} {n}')

# Build labeled rows.
labeled = [r for r in (build_row(i) for i in raw_issues) if r is not None]
class_counts = Counter(r['label'] for r in labeled)
print(f'\nAfter resolve_label + merge ({len(labeled)} usable issues):')
for c in CLASS_NAMES:
    print(f'  {c:<10} {class_counts[c]}')

## 3. Stratified time-aware split (70/15/15)

Most recent 15% by `closed_at` → **test** (prevents temporal leakage; the model must generalise to future-shaped issues).
Remaining 85% → stratified by class → **train (82.4%) + val (17.6%)** of the remainder = ~15% of total.

In [ ]:
import random
from collections import defaultdict

RANDOM_SEED = 42
TEST_FRACTION = 0.15
VAL_OF_REMAINING = 0.15 / 0.85

rng = random.Random(RANDOM_SEED)
labeled_sorted = sorted(labeled, key=lambda r: r['closed_at'])

# Time-aware test cut.
test_cut = int(len(labeled_sorted) * (1 - TEST_FRACTION))
remaining, test = labeled_sorted[:test_cut], labeled_sorted[test_cut:]

# Verify temporal invariant.
assert max(r['closed_at'] for r in remaining) <= min(r['closed_at'] for r in test), \
    'temporal leakage: train/val overlaps test in time'

# Stratified val sample from remaining.
by_class: dict[str, list[dict[str, Any]]] = defaultdict(list)
for r in remaining:
    by_class[r['label']].append(r)

train: list[dict[str, Any]] = []
val: list[dict[str, Any]] = []
for cls, rows in by_class.items():
    rng.shuffle(rows)
    n_val = max(1, int(len(rows) * VAL_OF_REMAINING))
    val.extend(rows[:n_val])
    train.extend(rows[n_val:])

rng.shuffle(train)
rng.shuffle(val)

for name, split in (('train', train), ('val', val), ('test', test)):
    cc = Counter(r['label'] for r in split)
    print(f'{name:<5} n={len(split):<5}  ' + '  '.join(f'{c}={cc[c]}' for c in CLASS_NAMES))

# Persist for reproducibility.
for name, split in (('train', train), ('val', val), ('test', test)):
    path = DATA_DIR / f'{name}.jsonl'
    with path.open('w') as fh:
        for row in split:
            fh.write(json.dumps(row) + '\n')
    print(f'saved → {path}')

## 4. Tokenize

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def to_hf(split: list[dict[str, Any]]) -> Dataset:
    ds = Dataset.from_dict({
        'text': [r['text'] for r in split],
        'label': [r['label_idx'] for r in split],
    })
    return ds.map(
        lambda batch: tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH, padding=False),
        batched=True,
    )


train_ds = to_hf(train)
val_ds = to_hf(val)
print(train_ds, val_ds, sep='\n')

## 5. Train DistilBERT (3-class head, EarlyStopping)

In [ ]:
import numpy as np
import evaluate
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

OUTPUT_DIR = Path('/content/artifacts/classifier')
BEST_DIR = OUTPUT_DIR / 'best'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(CLASS_NAMES),
    id2label={i: c for i, c in enumerate(CLASS_NAMES)},
    label2id=CLASS_TO_IDX,
)

f1_metric = evaluate.load('f1')


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return f1_metric.compute(predictions=preds, references=labels, average='macro')


training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    report_to='none',
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

## 6. Save best checkpoint + write `model_card.json`

**The model card is the contract with the backend.** `load_classifier()` reads `classes`, `model_sha256`, `version`, `hyperparameters.max_length` and refuses to boot on any drift.

In [ ]:
import hashlib
from datetime import UTC

BEST_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(BEST_DIR))
tokenizer.save_pretrained(str(BEST_DIR))


def sha256_weights(directory: Path) -> str:
    """MUST match the algorithm in backend/app/ml/classifier.py::_sha256_dir_weights."""
    h = hashlib.sha256()
    matched: list[Path] = []
    for pat in ('*.safetensors', '*.bin'):
        matched.extend(sorted(directory.glob(pat)))
    for p in matched:
        with p.open('rb') as f:
            for chunk in iter(lambda: f.read(8192), b''):
                h.update(chunk)
    return f'sha256:{h.hexdigest()}'


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return f'sha256:{h.hexdigest()}'


eval_results = trainer.evaluate()
val_f1 = float(eval_results.get('eval_f1', 0.0))

model_sha256 = sha256_weights(BEST_DIR)
train_path = DATA_DIR / 'train.jsonl'
train_sha256 = sha256_file(train_path)

model_card = {
    'architecture': MODEL_NAME,
    'num_labels': len(CLASS_NAMES),
    'classes': list(CLASS_NAMES),
    'class_to_idx': CLASS_TO_IDX,
    'hyperparameters': {
        'learning_rate': training_args.learning_rate,
        'per_device_train_batch_size': training_args.per_device_train_batch_size,
        'num_train_epochs': training_args.num_train_epochs,
        'weight_decay': training_args.weight_decay,
        'warmup_ratio': training_args.warmup_ratio,
        'max_length': MAX_LENGTH,
    },
    'freeze_policy': 'all layers unfrozen — full fine-tune of DistilBERT',
    'training_data_sha256': train_sha256,
    'training_data_size': {'train': len(train), 'val': len(val), 'test': len(test)},
    'metrics': {'val_f1_macro': val_f1, 'raw_eval': eval_results},
    'model_sha256': model_sha256,
    'trained_at': datetime.now(UTC).isoformat(),
    'version': '1.0.0',
    'dataset_repo': REPO,
}

(BEST_DIR / 'model_card.json').write_text(json.dumps(model_card, indent=2))
print(json.dumps(model_card, indent=2))

## 7. Persist artifacts to Drive (zip + copy)

In [ ]:
import shutil

# Copy the unzipped best/ to Drive — what load_classifier() will read.
drive_best = ARTIFACTS_DIR / 'best'
if drive_best.exists():
    shutil.rmtree(drive_best)
shutil.copytree(BEST_DIR, drive_best)

# Also zip for portability (download / scp / upload to MinIO).
zip_base = ARTIFACTS_DIR / 'classifier_best'
shutil.make_archive(str(zip_base), 'zip', root_dir=BEST_DIR.parent, base_dir=BEST_DIR.name)

print('Persisted to Drive:')
for p in sorted(drive_best.iterdir()):
    print(f'  {p.name:<30} {p.stat().st_size:>12} bytes')
print(f'\nZip archive: {zip_base}.zip ({Path(str(zip_base) + ".zip").stat().st_size} bytes)')

## 8. Round-trip verify — simulate `load_classifier()`

This re-runs **the exact checks** the backend will perform at boot. If this cell fails, the backend will refuse to start with the same error. Run it before you commit the artifact.

In [ ]:
EXPECTED_CLASSES = ('bug', 'feature', 'support')  # backend contract

card = json.loads((drive_best / 'model_card.json').read_text())

assert 'classes' in card, 'card missing classes field'
assert tuple(card['classes']) == EXPECTED_CLASSES, (
    f'class-set drift: card={card["classes"]} vs backend={list(EXPECTED_CLASSES)}'
)

assert card.get('model_sha256', '').startswith('sha256:'), 'card missing model_sha256'

actual = sha256_weights(drive_best)
assert actual == card['model_sha256'], f'sha256 mismatch\n  expected: {card["model_sha256"]}\n  actual:   {actual}'

assert (drive_best / 'tokenizer.json').exists() or (drive_best / 'tokenizer_config.json').exists(), (
    'tokenizer files missing — AutoTokenizer.from_pretrained() will fail'
)

print('OK — artifact passes every check load_classifier() will run.')
print(f'   classes      : {card["classes"]}')
print(f'   val_f1_macro : {card["metrics"]["val_f1_macro"]:.4f}')
print(f'   model_sha256 : {card["model_sha256"][:24]}…')

## Deploy to backend

1. **Download** `artifacts/classifier/best/` from Drive (or fetch directly inside docker via gdown / boto3 / MinIO).
2. **Place** at `backend/artifacts/classifier/best/` (gitignored — never commit weights).
3. **Start the stack:** `docker-compose up`. The model-server lifespan will call `load_classifier(Path('artifacts/classifier/best'))` and verify the same checks Section 8 just passed.
4. **Smoke test:** `curl -X POST localhost:8000/classify -d '{"text":"crash on empty tensor"}'` → `{"label":"bug", ...}`.